In [ ]:
from fabrictestbed_extensions.fablib.fablib import FablibManager as fablib_manager
fablib = fablib_manager(project_id="f5b4fc7d-978f-45be-9530-b38db8ef5046")
slice_name = 'prediction_market_project_4-20-26'

In [ ]:
# Create the nodes and network interfaces
image = 'default_ubuntu_22'
site = "LOSA" 

slice = fablib.new_slice(name=slice_name)

# Trader 1
trader1 = slice.add_node(name="trader1", image=image, cores=2, ram=4, disk=10, site=site)
trader1_iface = trader1.add_component(model="NIC_Basic", name="iface1").get_interfaces()[0]

# Trader 2
trader2 = slice.add_node(name="trader2", image=image, cores=2, ram=4, disk=10, site=site)
trader2_iface = trader2.add_component(model="NIC_Basic", name="iface1").get_interfaces()[0]

# Gateway
gateway = slice.add_node(name="gateway", image=image, cores=2, ram=4, disk=10, site=site)
gateway_iface = gateway.add_component(model="NIC_Basic", name="iface1").get_interfaces()[0]

# Engine 1
engine1 = slice.add_node(name="engine1", image=image, cores=2, ram=4, disk=10, site=site)
engine1_iface = engine1.add_component(model="NIC_Basic", name="iface1").get_interfaces()[0]

# Engine 2
engine2 = slice.add_node(name="engine2", image=image, cores=2, ram=4, disk=10, site=site)
engine2_iface = engine2.add_component(model="NIC_Basic", name="iface1").get_interfaces()[0]

# Connect the 5 interfaces to a single network
net = slice.add_l2network(name="net", interfaces=[trader1_iface, trader2_iface, gateway_iface, engine1_iface, engine2_iface], type="L2Bridge")

slice.submit()

In [ ]:
slice = fablib.get_slice(slice_name)
slice.list_nodes()
slice.list_interfaces()

In [ ]:
# get nodes and interfaces if slice created in previous session
trader1 = slice.get_node(name="trader1")
trader2 = slice.get_node(name="trader2")
gateway = slice.get_node(name="gateway")
engine1 = slice.get_node(name="engine1")
engine2 = slice.get_node(name="engine2")

trader1_iface = slice.get_node(name="trader1").get_interfaces()[0]
trader2_iface = slice.get_node(name="trader2").get_interfaces()[0]
gateway_iface = slice.get_node(name="gateway").get_interfaces()[0]
engine1_iface = slice.get_node(name="engine1").get_interfaces()[0]
engine2_iface = slice.get_node(name="engine2").get_interfaces()[0]

In [ ]:
# Install C++ and QuickFIX
install_cmd = "sudo apt-get update -y -qq && sudo apt-get install -y build-essential cmake libquickfix-dev"

# Execute on each node
for node in [trader1, trader2, gateway, engine1, engine2]:
    node.execute(install_cmd)

In [ ]:
from ipaddress import ip_network

# Define local subnet
subnet = ip_network('192.168.1.0/24')

# Assign static IPs, keep same as .cfg files
trader1_iface.ip_addr_add(addr='192.168.1.10', subnet=subnet)
trader2_iface.ip_addr_add(addr='192.168.1.14', subnet=subnet)
gateway_iface.ip_addr_add(addr='192.168.1.11', subnet=subnet)
engine1_iface.ip_addr_add(addr='192.168.1.12', subnet=subnet)
engine2_iface.ip_addr_add(addr='192.168.1.13', subnet=subnet)

# Bring the network links up
trader1_iface.ip_link_up()
trader2_iface.ip_link_up()
gateway_iface.ip_link_up()
engine1_iface.ip_link_up()
engine2_iface.ip_link_up()

print("network interfaces active")

In [ ]:
#Test that gateway has communication with everyone
print("pinging Trader 1 from Gateway...")
gateway.execute("ping -c 3 192.168.1.10")

print("\npinging Trader 2 from Gateway...")
gateway.execute("ping -c 3 192.168.1.14")

print("\npinging Engine 1 from Gateway...")
gateway.execute("ping -c 3 192.168.1.12")

print("\npinging Engine 2 from Gateway...")
gateway.execute("ping -c 3 192.168.1.13")

In [26]:
proj_dir = "PredictionMarketMatchingEngine"

# Upload project directory to each node
print("Starting Uploads")
trader1.upload_directory(".", f"./{proj_dir}")
trader2.upload_directory(".", f"./{proj_dir}")
gateway.upload_directory(".", f"./{proj_dir}")
engine1.upload_directory(".", f"./{proj_dir}")
engine2.upload_directory(".", f"./{proj_dir}")
print("Uploads Complete")

# Compile the code on each node
print("Starting Compilation")
engine1.execute(f"cd {proj_dir} && make clean && make engine")
engine2.execute(f"cd {proj_dir} && make clean && make engine")
gateway.execute(f"cd {proj_dir} && make clean && make gateway")
trader1.execute(f"cd {proj_dir} && make clean && make trader")
trader2.execute(f"cd {proj_dir} && make clean && make trader")
print("Compilation Complete")

Starting Uploads
Uploads Complete
Starting Compilation
rm -f trader gateway engine
g++ -std=c++14 -Wall -g -Wextra -o engine engine.cpp -lquickfix -lpthread
engine.cpp:87:9: warning: dynamic exception specifications are deprecated in C++11 [-Wdeprecated]
   87 |         throw(FIX::FieldNotFound, FIX::IncorrectDataFormat, FIX::IncorrectTagValue, FIX::UnsupportedMessageType) override {
      |         ^~~~~
engine.cpp:201:54: warning: dynamic exception specifications are deprecated in C++11 [-Wdeprecated]
  201 |     void toApp(FIX::Message&, const FIX::SessionID&) throw(FIX::DoNotSend) override {}
      |                                                      ^~~~~
engine.cpp:202:64: warning: dynamic exception specifications are deprecated in C++11 [-Wdeprecated]
  202 |     void fromAdmin(const FIX::Message&, const FIX::SessionID&) throw(FIX::FieldNotFound, FIX::IncorrectDataFormat, FIX::IncorrectTagValue, FIX::RejectLogon) override {}
      |                                             

In [25]:
# kill all processes
nodes = [engine1, engine2, gateway, trader1, trader2]

for node in nodes:
    node.execute("killall -9 engine gateway trader || true")

print("all processes successfully killed")

gateway: no process found
trader: no process found
gateway: no process found
trader: no process found
engine: no process found
trader: no process found
engine: no process found
gateway: no process found
trader: no process found
engine: no process found
gateway: no process found
trader: no process found
all processes successfully killed


In [ ]:
#fablib.delete_slice(slice_name)